# The failure line, checked on validation (card #94)

cristiano's notebook for card #94, part of #33. The straight line from P(predecessor) to the failure rate is fitted on the training companies (formed January to September 2021). Does it predict how often the validation companies (formed October to December 2021) really fail?

It runs the steps of `jonas_real.ipynb` (the label, blocking, the matcher, P(predecessor), the line) on the whole crawl instead of a sample, with the same seed and the same costs, and changes nothing to make the result look better.

**The test set stays closed.** The 2022 companies are only counted: their own files (profile, insolvency, officers) are dropped before the adapter reads anything, and the matcher learns only from records at companies formed by September 2021.

**Masking:** the notebook prints counts and rates only, never the name of a person or a company.

This project is a prototype and is not financial advice.

In [1]:
# Setup: pick up changes to src/ without restarting, and make `import src` work from notebooks/
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

import random
import time
import warnings
from pathlib import PurePosixPath

import numpy as np
import pandas as pd

from src import decide, evaluate, real_slice, resolve
from src import ingest_companies_house as adapter

SEED = 42              # one seed for everything random, as in main.ipynb and jonas_real.ipynb
random.seed(SEED)
np.random.seed(SEED)
COSTS = (10, 2, 1)     # the team's costs: a missed case, a wrong flag, asking

## The label
The crawl's 2021 query companies go through the team's adapter and get the team's label; the 2022 companies' own files are dropped first, so they are only counted.

In [2]:
raw = adapter.load_raw()   # every file in data/raw, as real_slice.load_slice reads them
done = real_slice.query_companies()
numbers = list(done.loc[done["year"] == "2021", "company_number"])
test_numbers = set(done.loc[done["year"] == "2022", "company_number"])
assert not test_numbers & set(numbers) and len(numbers) + len(test_numbers) == len(done)
print(f"query companies: {len(done)} | 2021: {len(numbers)} | 2022 (test, counted only): {len(test_numbers)}")

# The 2022 companies' own files (profile, insolvency, officers) go before the adapter reads anything
raw = {path: data for path, data in raw.items()
       if not (PurePosixPath(path).parent.name in ("company", "insolvency", "officers")
               and PurePosixPath(path).stem in test_numbers)}
with warnings.catch_warnings(record=True) as caught:   # the adapter's warnings, without the company numbers in them
    warnings.simplefilter("always")
    mentions, claims, labels, left_out = real_slice.slice_from_raw(raw, numbers)
for warning in caught:
    print("adapter warning:", str(warning.message).split(":")[0])
print("left out (no label):", pd.Series(left_out).value_counts().to_dict() if left_out else "none")
print("labelled 2021 query companies by split:", labels["split"].value_counts(dropna=False).to_dict())
print("mentions:", mentions["kind"].value_counts().to_dict(), "| claims:", len(claims))

query companies: 13761 | 2021: 6914 | 2022 (test, counted only): 6847


adapter warning: 6 companies have has_insolvency_history but no insolvency file
adapter warning: 27 director records skipped
left out (no label): none
labelled 2021 query companies by split: {'train': 5362, 'validation': 1552}
mentions: {'company': 102015, 'person': 65806} | claims: 65806


The label counts: how many companies failed under the team's label (insolvent or dissolved within 24 months), and how many of those failed through liquidation or administration, which is the insolvency-only label.

In [3]:
labels = labels[labels["split"].isin(["train", "validation"])].reset_index(drop=True)
# The team's label (insolvent or dissolved within 24 months), and insolvency only (count_dissolved=False)
labels["failed_insolvency"] = evaluate.label_failures(mentions, count_dissolved=False).loc[labels["company_id"]].to_numpy()
assert not (labels["failed_insolvency"] & ~labels["failed"]).any()   # insolvency only is part of the team's label
train = labels[labels["split"] == "train"]
validation = labels[labels["split"] == "validation"]

for name, part in [("train", train), ("validation", validation)]:
    n, failed, insolvent = len(part), int(part["failed"].sum()), int(part["failed_insolvency"].sum())
    print(f"{name}: {n} companies, {failed} failed ({failed / n:.1%}) | through liquidation or administration: "
          f"{insolvent} ({insolvent / n:.2%}) | dissolved only: {failed - insolvent}")

train: 5362 companies, 1778 failed (33.2%) | through liquidation or administration: 15 (0.28%) | dissolved only: 1763
validation: 1552 companies, 527 failed (34.0%) | through liquidation or administration: 1 (0.06%) | dissolved only: 526


## P(predecessor)
Blocking proposes the pairs of person records worth comparing: the same surname and first initial, or the same birth month and year and a similar surname.

In [4]:
candidates = resolve.candidate_pairs(mentions)
print(f"{len(candidates)} candidate pairs from {int((mentions['kind'] == 'person').sum())} person records")

263601 candidate pairs from 65806 person records


The matcher learns from training pairs only, and only from pairs whose two records are at companies formed by September 2021. Then it gives every candidate pair its `p`, the chance that the two records are one person.

In [5]:
splits = real_slice.person_split(mentions, claims, labels)   # each person record's split, from the company it directs
labelled = resolve.labelled_pairs(mentions)                  # same officer id: label 1; a candidate pair with different ids: 0
labelled["split"] = real_slice.pair_split(labelled, splits)
later = (labelled["split"] == "train") & ~real_slice.train_period_pairs(labelled, mentions, claims)
labelled.loc[later, "split"] = "left out"                    # a record at a company formed after September 2021
train_pairs = labelled[labelled["split"] == "train"].reset_index(drop=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = resolve.train_matcher(resolve.pair_features(train_pairs, mentions), train_pairs["label"], SEED)
for warning in caught:
    print("matcher warning:", str(warning.message)[:200])
print("training pairs by label:", train_pairs["label"].value_counts().sort_index().to_dict())
print("matcher weights:", resolve.matcher_weights(model).round(2).to_dict())

scored = resolve.scored_pairs(model, candidates, mentions)   # every candidate pair, with the matcher's p

training pairs by label: {0: 145518, 1: 102213}
matcher weights: {'name_similarity': 17.88, 'name_missing': 0.0, 'birth_same': 6.16, 'birth_missing': 0.13, 'postcode_same': 1.85, 'postcode_missing': 1.2, 'nationality_same': 1.94, 'nationality_missing': -0.08, 'intercept': -24.94}


Each company's matches are the pairs that tie one of its directors on its formation date to someone who ran a company insolvent by then (`decide.candidate_matches`), and P(predecessor) = 1 − ∏(1 − p) over them. `matches_for` hands `candidate_matches` only the rows that can change its answer, because the full tables make every call much slower: compare the time printed here with the one the check prints.

In [6]:
# A match is a pair from a director record of the company to a record at an insolvent company. So for one
# company only these rows can change candidate_matches' answer: the pairs that touch one of its director
# records, the claims at the company plus the claims of those pairs' records (each person record has
# exactly one claim), and the companies those claims point at.
start = time.perf_counter()
companies = mentions[mentions["kind"] == "company"]
assert companies["company_number"].is_unique and claims["subject"].is_unique
mention_of = companies.set_index("company_number")["mention_id"]
records_at = claims.groupby("object")["subject"].apply(list)   # a company's person records, any date
rows_of = {}                                                    # a person record's rows in `scored`
for row, (a, b) in enumerate(zip(scored["a"], scored["b"])):
    rows_of.setdefault(a, []).append(row)
    rows_of.setdefault(b, []).append(row)


def matches_for(query):
    """decide.candidate_matches(query, mentions, claims, scored), given only the rows that matter."""
    mention = mention_of[query]
    rows = sorted({row for record in records_at.get(mention, []) for row in rows_of.get(record, [])})
    pairs = scored.iloc[rows]                                   # same order as in `scored`
    ends = set(pairs["a"]) | set(pairs["b"])
    some_claims = claims[(claims["object"] == mention) | claims["subject"].isin(ends)]
    some_companies = companies[companies["mention_id"].isin(set(some_claims["object"]) | {mention})]
    return decide.candidate_matches(query, some_companies, some_claims, pairs)


matches = {query: matches_for(query) for query in labels["company_id"]}
p_pred = pd.Series({query: decide.company_p_predecessor(found) for query, found in matches.items()}, dtype=float)
print(f"matches and P(predecessor) for all {len(matches)} companies in {(time.perf_counter() - start) / 60:.1f} minutes")

matches and P(predecessor) for all 6914 companies in 2.0 minutes


The check: for 30 random companies and 30 with P(predecessor) above 0, the full call must give exactly the same matches as `matches_for`.

In [7]:
rng = np.random.default_rng(SEED)
positive = p_pred.index[p_pred > 0]
checked = [str(query) for query in list(rng.choice(labels["company_id"], 30, replace=False))
           + list(rng.choice(positive, min(30, len(positive)), replace=False))]
start = time.perf_counter()
for query in checked:
    full = decide.candidate_matches(query, mentions, claims, scored)
    assert (full is None) == (matches[query] is None), query
    if full is not None:
        pd.testing.assert_frame_equal(full, matches[query])
per_call = (time.perf_counter() - start) / len(checked)
print(f"check: matches_for equals candidate_matches on the full tables for all {len(checked)} companies checked "
      f"({sum(matches[q] is not None and not matches[q].empty for q in checked)} of them with at least one match)")
print(f"the full call takes {per_call:.2f} s a company: about {per_call * len(labels) / 60:.0f} minutes for all {len(labels)}")

check: matches_for equals candidate_matches on the full tables for all 60 companies checked (42 of them with at least one match)
the full call takes 0.17 s a company: about 20 minutes for all 6914


How many companies have a likely insolvent predecessor, in each split.

In [8]:
for name, part in [("train", train), ("validation", validation)]:
    p = p_pred.loc[part["company_id"]]
    print(f"{name}: = 0: {int((p == 0).sum())} | > 0: {int((p > 0).sum())} | >= 0.5: {int((p >= 0.5).sum())} | "
          f"no director records: {int(p.isna().sum())} | of {len(p)}")

train: = 0: 3511 | > 0: 1835 | >= 0.5: 351 | no director records: 16 | of 5362
validation: = 0: 1065 | > 0: 486 | >= 0.5: 105 | no director records: 1 | of 1552


## The line on train, and the thresholds
The straight line from P(predecessor) to the failure rate, fitted on the training companies (`decide.fit_failure_line`); the thresholds the costs give; and, on the companies the line was fitted on, its prediction next to the real failure rate in three groups of P(predecessor).

In [9]:
clear_below, flag_above = decide.thresholds(*COSTS)


def fit_line(column):
    """decide.fit_failure_line on the training companies for one label, printed with its slope before clipping."""
    x = p_pred.loc[train["company_id"]]
    line = decide.fit_failure_line(x, train[column])
    keep = x.notna().to_numpy()
    slope, intercept = np.polyfit(x.to_numpy()[keep], train[column].to_numpy()[keep].astype(float), 1)   # before clipping
    print(f"line fitted on {int(keep.sum())} training companies ({int(train[column][keep].sum())} positives): "
          f"rate_without = {line[0]:.1%}, rate_with = {line[1]:.1%} | slope {slope:+.4f} "
          f"({'down' if slope < 0 else 'up' if slope > 0 else 'flat'}), intercept {intercept:.4f} before clipping")
    print(f"costs {COSTS}: CLEAR at or below {clear_below:.0%}, FLAG at or above {flag_above:.0%}")
    low, high = min(line), max(line)   # every P(fail) the line gives lies between its two ends
    print(f"P(fail) on this line runs from {low:.1%} to {high:.1%}: FLAG {'can' if high >= flag_above else 'can never'} "
          f"fire, CLEAR {'can' if low <= clear_below else 'can never'} fire")
    return line


def by_group(part, column, line):
    """The line's prediction next to the real failure rate, in three groups of P(predecessor) (jonas_real.ipynb)."""
    check = pd.DataFrame({"p_predecessor": p_pred.loc[part["company_id"]].to_numpy(),
                          "failed": part[column].to_numpy()})
    check["predicted"] = [decide.p_fail(p, line) for p in check["p_predecessor"]]
    groups = pd.cut(check["p_predecessor"], [-0.01, 0, 0.5, 1], labels=["0", "above 0, up to 0.5", "above 0.5"])
    table = check.groupby(groups, observed=True).agg(companies=("failed", "size"), failed=("failed", "sum"),
                                                     predicted=("predicted", "mean"))
    table["predicted"] = table["predicted"].map("{:.1%}".format)
    table["really failed (95% Wilson interval)"] = [
        f"{k / n:.1%} ({low:.1%} to {high:.1%})"
        for k, n, (low, high) in ((int(k), int(n), real_slice.wilson(int(k), int(n)))
                                  for k, n in zip(table["failed"], table["companies"]))]
    known = check["p_predecessor"].notna()
    print(f"all {int(known.sum())} with director records: line predicts {check.loc[known, 'predicted'].mean():.1%} "
          f"on average, really failed {check.loc[known, 'failed'].mean():.1%}")
    return table


line = fit_line("failed")
by_group(train, "failed", line)

line fitted on 5346 training companies (1762 positives): rate_without = 34.1%, rate_with = 14.4% | slope -0.1967 (down), intercept 0.3411 before clipping
costs (10, 2, 1): CLEAR at or below 10%, FLAG at or above 50%
P(fail) on this line runs from 14.4% to 34.1%: FLAG can never fire, CLEAR can never fire
all 5346 with director records: line predicts 33.0% on average, really failed 33.0%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,3511,1235,34.1%,35.2% (33.6% to 36.8%)
"above 0, up to 0.5",1484,466,34.0%,31.4% (29.1% to 33.8%)
above 0.5,351,61,17.1%,17.4% (13.8% to 21.7%)


## The check on validation (#94)
The same table for the validation companies, which the line never saw. Where the prediction lies outside the 95% interval, the line is wrong for that group.

In [10]:
by_group(validation, "failed", line)

all 1551 with director records: line predicts 32.9% on average, really failed 33.9%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,1065,407,34.1%,38.2% (35.3% to 41.2%)
"above 0, up to 0.5",381,105,34.0%,27.6% (23.3% to 32.3%)
above 0.5,105,14,17.1%,13.3% (8.1% to 21.1%)


## The insolvency-only label
The same line and tables when only liquidation or administration counts as failing (the label before card #142 added dissolution).

In [11]:
line_insolvency = fit_line("failed_insolvency")
for name, part in [("train", train), ("validation", validation)]:
    print(f"-- {name}:")
    display(by_group(part, "failed_insolvency", line_insolvency))

line fitted on 5346 training companies (15 positives): rate_without = 0.3%, rate_with = 0.3% | slope +0.0005 (up), intercept 0.0028 before clipping
costs (10, 2, 1): CLEAR at or below 10%, FLAG at or above 50%
P(fail) on this line runs from 0.3% to 0.3%: FLAG can never fire, CLEAR can fire
-- train:
all 5346 with director records: line predicts 0.3% on average, really failed 0.3%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,3511,10,0.3%,0.3% (0.2% to 0.5%)
"above 0, up to 0.5",1484,4,0.3%,0.3% (0.1% to 0.7%)
above 0.5,351,1,0.3%,0.3% (0.1% to 1.6%)


-- validation:
all 1551 with director records: line predicts 0.3% on average, really failed 0.1%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,1065,1,0.3%,0.1% (0.0% to 0.5%)
"above 0, up to 0.5",381,0,0.3%,0.0% (0.0% to 1.0%)
above 0.5,105,0,0.3%,0.0% (0.0% to 3.5%)


**What it shows (AI draft, the owner puts it in their own words)**

- **Why the line slopes down.** On the training companies, a likely insolvent predecessor goes with *fewer* failures, not more: in the train table, the companies above 0.5 fail less often than those at 0. The line is the straight line closest to those points, so it falls, from 34.1% with no likely predecessor to 14.4% with a certain one. Why the companies behave like this is not tested here. One guess: almost every failure under the team's label is a dissolution (see the label counts), and a director who has run a company before may less often start one that is struck off within two years.
- **FLAG can never fire, and CLEAR can't either.** Every P(fail) the line gives lies between its two ends, 14.4% and 34.1% (the line printed under the fit). At costs 10/2/1 the decision layer clears at 10% or less and flags at 50% or more, so with this label and this line it answers NOT ENOUGH EVIDENCE for every company.
- **The line misses the two lower groups on validation.** In the #94 table, the prediction for P(predecessor) = 0 lies below the interval of what really happened, and the prediction for the middle group lies above its interval; only the group above 0.5 has its prediction inside. The middle group's prediction is just outside its interval on train too: those companies' P(predecessor) is close to 0 on average, so the line predicts nearly the same as at 0, yet they fail less. A straight line can't show a drop that comes with any match at all, however small its `p`. On average the line looks right (the "all ... with director records" lines), because its misses go in opposite directions.
- **Insolvency only has too few failures to say anything.** Through liquidation or administration, 15 training companies and 1 validation company failed (see the label counts). The line fitted on those 15 is flat at 0.3%, and 1 failure can't check it: on validation every group's interval reaches down to 0.0%. With this line the decision layer would CLEAR every company with director records, whatever its directors ran before.
- **This is the project's main limit.** The decision layer was built to flag a company whose director likely ran an insolvent company before. With the team's label that link goes with fewer failures, so the tool can't flag anyone; with the insolvency-only label there are too few failures to learn from or check against. This is reported as found: the label, the costs and the line were not changed to make it look better.